# C1 - CORAL direct verification (the cold-start existence proof)
**Question.** What does CORAL actually achieve on a TRUE cold-start, and is the headline number clean? We ran the CORAL repo directly (not a re-implementation).

*Data: `mmpartnet_out/coral_reproduction.json` -- recorded from a CUDA GPU run; see its `provenance` (repo, env, protocol). Displayed here because recompute needs the CORAL repo + a GPU.*

## Definitions
- **component-wise split**: 0% protein AND 0% RNA overlap with train = the true cold-start.
- **Path A**: honest per-fold retrain (train only on that fold's train.csv).

In [1]:
import json; from pathlib import Path; from IPython.display import Markdown, display
d = json.loads((Path('..')/'..'/'mmpartnet_out'/'coral_reproduction.json').read_text(encoding='utf-8'))
p = d['provenance']
display(Markdown(f"**Computed on:** {p['computed_on']}  \n**Repo:** {p['repo']}  \n"
                 f"**Env:** {p['env']}  \n**Protocol:** {p['protocol_pathA']}"))
tbl = '| setting | metric | value | note |\n|---|---|---|---|\n'
for k, v in d['results'].items():
    tbl += f"| {k} | {v['metric']} | {v['value']} | {v['note'][:80]} |\n"
display(Markdown(tbl))

**Computed on:** a CUDA GPU  
**Repo:** github.com/MarioCatalano99/Coral, cloned + run directly (not re-implemented)  
**Env:** torch 2.8+cu128 (the pinned 2.6+cu124 lacked compatible GPU kernels -> 'no kernel image')  
**Protocol:** per-fold retrain: Data/datasets/component-wise/fold_{k}/{train,val}.csv, batch 4, lr 1e-6, bf16, LoRA rank 8; ESM-2-150M + DNABERT2-117M + bidirectional cross-attention

| setting | metric | value | note |
|---|---|---|---|
| released_checkpoint_reused_across_folds | F1 | 0.92 | LEAKS: 15-23% pair overlap across the released folds; own-fold clean still 0.94  |
| paper_reported | F1 | 0.65 | CORAL paper's component-wise F1 |
| clean_per_fold_retrain_pathA | F1 | 0.57 | HONEST reproduction: retrain per fold on that fold's train.csv only, evaluate th |
| external_eclip_through_coral | AUROC | ~chance | our Moyon eCLIP -> CORAL CSV (adapters/coral.py) run through CORAL: ~chance -- w |


## Conclusion
The released single checkpoint **leaks across folds** (F1 0.92); a **clean per-fold retrain reproduces ~0.57**, below the paper's 0.65. Our eCLIP routed through CORAL is ~chance (different task). Net: CORAL is a rigorous cold-start **existence proof** that interaction/family diversity enables generalization -- the motivation for our independent-family scaling (S1). It is NOT a drop-in baseline for eCLIP window-occupancy.